[← Volver al resumen del proyecto](index.qmd)

Este cuaderno analiza los avisos de empleo que fui juntando mientras buscaba mi primer trabajo como analista de datos. Vengo de la Sociología y de una diplomatura en Ciencia de Datos.

Para ordenar la búsqueda armé una extensión de Chrome que lee un aviso y me dice qué pide, qué me falta y qué tan lejos estoy. La extensión no es el tema de este análisis: es la herramienta que me dejó convertir la búsqueda en datos. Acá uso esos datos para dos cosas:

1. **Mirar mi propio proceso**: qué avisos elegí evaluar, qué criterio terminé usando y dónde la herramienta acertaba o no.
2. **Mirar el mercado**: qué piden de verdad los puestos de entrada en Argentina para un perfil como el mío.

> **Una advertencia antes de empezar.** Estos datos son una muestra: salen de mis búsquedas, en LinkedIn y en un período corto. No representan "el mercado laboral argentino". Sirven para describir el recorte del mercado que ve alguien que busca un primer puesto en datos desde Mar del Plata, y para pensar con eso.

## Metodología: cómo evalúa la extensión

El uso es simple: abro un aviso, selecciono todo el texto, hago **clic derecho → "Evaluar este aviso"** y se abre un panel al costado con:

- un **veredicto**: *Postulate*, *Alcanzable* o *Lejos*, con el motivo;
- los requisitos separados en **excluyentes**, **deseables** y los que **solo se mencionan** en las tareas;
- el **nivel** que piden en cada herramienta (básico, intermedio, avanzado) comparado con el mío;
- alertas sobre **años de experiencia**, puestos **senior encubiertos**, **carreras** pedidas, **idiomas** y **modalidad**.

La primera vez que un aviso pide algo que la extensión no sabe de mí, me lo pregunta y lo guarda en mi perfil, así que el perfil se arma solo a medida que la uso. Cada aviso queda guardado en una página, "Mis avisos", que se sincroniza con una hoja de Google Sheets: de ahí salen los datos de este análisis.

## Las fuentes

| Fuente | Qué es | Período |
|---|---|---|
| **Mis evaluaciones** | Avisos que elegí abrir y evaluar uno por uno con clic derecho, con sus requisitos desglosados | Octubre y noviembre de 2026 |
| **Búsquedas en LinkedIn** | Todos los avisos que devolvieron búsquedas de nivel inicial y prácticas, importados a la extensión con el texto completo y evaluados en lote con las mismas reglas | Avisos publicados en las dos semanas previas al 12 de noviembre de 2026 |

Mis búsquedas apuntaron principalmente a tres tipos de puesto: **análisis de datos**, **trabajos administrativos** y **recursos humanos**. Todo lo que aparece abajo (herramientas, carreras, requisitos) depende de ese recorte: otras búsquedas darían otro mapa.

La diferencia entre las dos fuentes importa: **mis evaluaciones están sesgadas por mi elección** (abría lo que me parecía cercano), mientras que **las búsquedas muestran todo lo que devuelve el buscador**. Compararlas dice algo sobre cómo uno filtra sin darse cuenta.

In [1]:
import json, re, unicodedata
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

DATA = Path("data")
COLOR = {"Buena chance": "#1d7a5f", "Aplicable": "#d39b2a", "No": "#b4443c",
         "Postulate": "#1d7a5f", "Alcanzable": "#d39b2a", "Lejos": "#b4443c"}
px.defaults.template = "simple_white"
px.defaults.height = 420

def norm(s):
    # Minúsculas y sin tildes, para buscar palabras en los textos
    return unicodedata.normalize("NFD", str(s or "")).encode("ascii", "ignore").decode().lower()

def pct(a, b):
    return f"{100 * a / b:.0f}%" if b else "—"

In [2]:
mis = pd.read_csv(DATA / "mis_avisos.csv")
req = pd.read_csv(DATA / "mis_requisitos.csv")
tanda = pd.DataFrame(json.loads((DATA / "tanda_linkedin.json").read_text(encoding="utf-8")))
meta = json.loads((DATA / "meta.json").read_text(encoding="utf-8"))

tanda["texto_n"] = tanda["texto"].map(norm)
abiertos = tanda[tanda["estado_linkedin"] == "LISTED"].copy()

resumen = pd.DataFrame({
    "fuente": ["Mis evaluaciones", "Búsquedas en LinkedIn (sin repetidos)", "Búsquedas en LinkedIn (siguen abiertos)"],
    "avisos": [len(mis), len(tanda), len(abiertos)],
    "período": [f"{mis['fecha'].min()} a {mis['fecha'].max()}",
                f"publicados {tanda['publicado'].replace('', None).min()} a {tanda['publicado'].max()}", ""],
})
resumen

,fuente,avisos,período
0,Mis evaluaciones,257,2026-10-13 a 2026-11-12
1,Búsquedas en LinkedIn (sin repetidos),615,publicados 2026-10-31 a 2026-11-12
2,Búsquedas en LinkedIn (siguen abiertos),540,


## 1. El embudo

La primera pregunta es la más simple: de todo lo que aparece cuando busco "nivel inicial", ¿cuánto es realmente para mí?

Para responderla clasifiqué cada aviso de las búsquedas en tres niveles. **No** es que no aplica: está cerrado, es senior, pide 3 años o más (o 2 en un área que no es la mía), pide una carrera dura, es de ventas o está fuera de Buenos Aires o Mar del Plata sin ser remoto. **Aplicable** es que podría postularme, pero piden experiencia específica que no tengo o carreras puntuales que no son la mía. **Buena chance** es que cumplo lo que piden.

El veredicto de la extensión no alcanzaba para separar *aplicable* de *buena chance*, así que definí ese criterio marcando a mano una muestra de avisos y anotando por qué. Por ejemplo: "tengo los conocimientos, pero piden experiencia en esa tarea" o "piden carreras económicas". Después apliqué ese criterio al resto.

In [3]:
n_encontrados = meta["avisos_encontrados"]
n_unicos = len(tanda)
n_abiertos = len(abiertos)
n_pasan = int((tanda["nivel_calibrado"] != "No").sum())
n_buena = int((tanda["nivel_calibrado"] == "Buena chance").sum())

fig = go.Figure(go.Funnel(
    y=["Avisos que devolvió la búsqueda", "Sin repetidos", "Todavía abiertos",
       "Pasan los filtros duros", "Buena chance"],
    x=[n_encontrados, n_unicos, n_abiertos, n_pasan, n_buena],
    textinfo="value+percent initial",
    marker={"color": ["#9aa5b1", "#9aa5b1", "#7b8794", "#d39b2a", "#1d7a5f"]},
))
fig.update_layout(title="De la búsqueda a los avisos con chances reales", height=380)
fig.show()

display(Markdown(
    f"De **{n_encontrados}** avisos que devolvieron las búsquedas de nivel inicial, "
    f"solo **{n_buena}** ({pct(n_buena, n_encontrados)}) quedaron como buena chance."
))

De **663** avisos que devolvieron las búsquedas de nivel inicial, solo **53** (8%) quedaron como buena chance.

### ¿Por qué se cae un aviso?

Un mismo aviso puede caerse por varias razones a la vez. El gráfico cuenta cada razón por separado.

In [4]:
def motivo_grupo(m):
    m = norm(m)
    if "cerrado" in m: return "El aviso ya cerró"
    if "nivel publicado" in m or "senior" in m or "jefatura" in m: return "Es senior o de liderazgo"
    if "anos" in m: return "Pide más años de experiencia"
    if "carrera" in m or "contador" in m: return "Pide otra carrera"
    if "ventas" in m or "comision" in m: return "Es de ventas"
    if "fuera de ba" in m or "otro pais" in m or "region" in m: return "Ubicación o país"
    if "ingenieria" in m or "desarrollo" in m: return "Es de ingeniería o desarrollo"
    if "idioma" in m: return "Pide otro idioma"
    return "Otros"

caidos = (tanda[tanda["nivel_calibrado"] == "No"]
          .explode("motivos")
          .assign(grupo=lambda d: d["motivos"].map(motivo_grupo))
          .drop_duplicates(["id", "grupo"]))
conteo = caidos["grupo"].value_counts().rename_axis("motivo").reset_index(name="avisos")

fig = px.bar(conteo.sort_values("avisos"), x="avisos", y="motivo", orientation="h",
             title="Motivos por los que un aviso queda afuera (búsquedas en LinkedIn)",
             color_discrete_sequence=["#b4443c"], text="avisos")
fig.update_layout(yaxis_title="", xaxis_title="avisos")
fig.show()

## 2. El "junior" que no es junior

LinkedIn le pone a cada aviso un nivel (prácticas, nivel inicial, intermedio...). Lo publica la empresa. Mi pregunta: entre los avisos que la empresa llamó de **nivel inicial o prácticas**, ¿cuántos piden años de experiencia?

In [5]:
YEARS = re.compile(
    r"(\d{1,2})\s*\+?\s*(?:-\s*\d+\s*)?(?:anos|ano|years?)\b[^.\n]{0,60}(?:experiencia|experience)"
    r"|(?:experiencia|experience)[^.\n]{0,40}?(\d{1,2})\s*\+?\s*(?:anos|ano|years?)"
)

def anios_pedidos(texto):
    ys = [int(a or b) for a, b in YEARS.findall(texto) if 0 < int(a or b) < 11]
    return max(ys) if ys else None

abiertos["anios"] = abiertos["texto_n"].map(anios_pedidos)
entrada = abiertos[abiertos["nivel_linkedin"].isin(["Entry level", "Internship"])].copy()

def tramo(a):
    if pd.isna(a): return "No dice"
    if a <= 1: return "Hasta 1 año"
    if a == 2: return "2 años"
    return "3 años o más"

entrada["tramo"] = entrada["anios"].map(tramo)
orden = ["No dice", "Hasta 1 año", "2 años", "3 años o más"]
t = entrada["tramo"].value_counts().reindex(orden).fillna(0).astype(int).reset_index()
t.columns = ["experiencia pedida", "avisos"]

fig = px.bar(t, x="experiencia pedida", y="avisos", text="avisos",
             title="Avisos de nivel inicial o prácticas: cuánta experiencia piden",
             color="experiencia pedida",
             color_discrete_map={"No dice": "#9aa5b1", "Hasta 1 año": "#1d7a5f",
                                 "2 años": "#d39b2a", "3 años o más": "#b4443c"})
fig.update_layout(showlegend=False, xaxis_title="")
fig.show()

n_e = len(entrada)
n_2 = int((entrada["anios"] >= 2).sum())
n_3 = int((entrada["anios"] >= 3).sum())
display(Markdown(
    f"De **{n_e}** avisos marcados por la empresa como nivel inicial o prácticas, "
    f"**{n_2} ({pct(n_2, n_e)}) piden 2 años o más** y {n_3} ({pct(n_3, n_e)}) piden 3 o más."
))

De **91** avisos marcados por la empresa como nivel inicial o prácticas, **29 (32%) piden 2 años o más** y 22 (24%) piden 3 o más.

## 3. Credenciales: ¿qué carreras piden?

Como sociólogo, esto es lo que más me interesaba. Muchas tareas de análisis de datos (armar reportes, leer indicadores, diseñar encuestas) son las mismas que se aprenden en ciencias sociales. ¿Qué carreras aparecen en los requisitos?

Para no contar menciones sueltas (por ejemplo "business" en la descripción de la empresa), solo miro **las oraciones que hablan de estudios o formación**.

In [6]:
EDU = re.compile(r"(estudiante|graduad|licenciad|recibid|carrera|titulo|degree|bachelor|student|egresad|pursuing|formacion)")
GRUPOS = {
    "Económicas / administración": r"econom|administracion de empresas|administracion|contador|contab|finanz|finance|business|negocios|comercializacion",
    "Ingenierías / sistemas": r"ingenier|engineering|sistemas|informatica|computer science|computacion",
    "Marketing / comunicación": r"marketing|publicidad|comunicacion|communications?|periodismo|relaciones publicas",
    "Exactas / estadística / ciencia de datos": r"matematic|estadistic|statistics|actuar|ciencia de datos|data science|fisica|mathematics",
    "Psicología / RRHH": r"psicolog|psychology|recursos humanos|human resources|relaciones (?:del trabajo|laborales)",
    "Salud / biológicas": r"medicina|biolog|bioquim|farmac|life sciences|enfermer|nursing",
    "Ciencias sociales / sociología": r"sociolog|ciencias sociales|social sciences?|ciencia politica|antropolog",
    "Cualquier carrera": r"cualquier carrera|any (?:field|discipline|background)|all degree|todas las carreras",
}

def oraciones_formacion(texto):
    partes = re.split(r"(?<=[.;•·!?])\s+|\n+|\s[-–•✅]\s", texto)
    return " ".join(p for p in partes if EDU.search(p))

abiertos["formacion"] = abiertos["texto_n"].map(oraciones_formacion)
con_carrera = abiertos[abiertos["formacion"].str.len() > 0]
filas = [{"carreras": g, "avisos": int(con_carrera["formacion"].str.contains(p).sum())} for g, p in GRUPOS.items()]
carreras = pd.DataFrame(filas).sort_values("avisos")
carreras["color"] = carreras["carreras"].map(lambda g: "#1d7a5f" if g in ("Ciencias sociales / sociología", "Cualquier carrera") else "#7b8794")

fig = px.bar(carreras, x="avisos", y="carreras", orientation="h", text="avisos",
             title=f"Carreras que aparecen en los requisitos ({len(con_carrera)} avisos abiertos que hablan de formación)")
fig.update_traces(marker_color=carreras["color"])
fig.update_layout(yaxis_title="", xaxis_title="avisos")
fig.show()

soc = int(con_carrera["formacion"].str.contains(GRUPOS["Ciencias sociales / sociología"]).sum())
eco = int(con_carrera["formacion"].str.contains(GRUPOS["Económicas / administración"]).sum())
display(Markdown(
    f"De los avisos que mencionan formación, **{eco}** nombran carreras económicas o de administración "
    f"y apenas **{soc}** nombran ciencias sociales o sociología."
))

De los avisos que mencionan formación, **159** nombran carreras económicas o de administración y apenas **6** nombran ciencias sociales o sociología.

## 4. Herramientas: qué piden

Acá uso **mis evaluaciones**, porque la extensión desglosó cada aviso en requisitos y distingue si cada uno es **excluyente**, **deseable** o si **solo se menciona** en las tareas.

> Estas herramientas salen de **mis búsquedas** (análisis de datos, trabajos administrativos y recursos humanos). No son "las herramientas más pedidas del mercado": son las más pedidas en los puestos a los que apunto.

In [7]:
TABLEROS = {"Herramienta de tableros": "Tableros (Power BI, Tableau…)"}
req["requisito"] = req["requisito"].replace(TABLEROS)
herr = req[req["fuente"] == "reglas"].copy()
top = herr["requisito"].value_counts().head(15).index
tabla = (herr[herr["requisito"].isin(top)]
         .groupby(["requisito", "tipo"]).size().reset_index(name="avisos"))
orden_req = herr["requisito"].value_counts().head(15).index[::-1]

fig = px.bar(tabla, x="avisos", y="requisito", color="tipo", orientation="h",
             category_orders={"requisito": list(orden_req), "tipo": ["excluyente", "deseable", "se menciona"]},
             color_discrete_map={"excluyente": "#3b4a5a", "deseable": "#8fa3b8", "se menciona": "#d5dde5"},
             title=f"Lo que más aparece en los {mis.shape[0]} avisos que evalué")
fig.update_layout(yaxis_title="", xaxis_title="avisos", legend_title="")
fig.show()

### ¿Qué tan excluyente es cada herramienta?

Que una herramienta aparezca mucho no significa que sea un filtro. Este gráfico muestra, para las herramientas más mencionadas, en qué proporción de los avisos que la nombran es **excluyente**. Las de arriba son las que, si faltan, más puertas cierran.

In [8]:
excl = (herr.groupby("requisito")
        .agg(avisos=("aviso_id", "nunique"), excluyente=("tipo", lambda t: (t == "excluyente").sum()))
        .query("avisos >= 12")
        .assign(pct=lambda d: (100 * d["excluyente"] / d["avisos"]).round(0))
        .sort_values("pct")
        .reset_index())

fig = px.bar(excl, x="pct", y="requisito", orientation="h", text=excl["pct"].astype(int).astype(str) + "%",
             hover_data={"avisos": True, "excluyente": True, "pct": False},
             title="Proporción de avisos en los que cada herramienta es excluyente",
             color_discrete_sequence=["#3b4a5a"])
fig.update_layout(yaxis_title="", xaxis_title="% de los avisos que la mencionan", xaxis_range=[0, 100])
fig.show()

### Del aviso al currículum (y a lo que me falta aprender)

Con muchos avisos guardados, la extensión arma dos listas a partir de mi perfil:

- **Qué destacar en el CV**: lo que más se pide y que ya manejo. Me sirvió para reordenar el currículum y poner arriba lo que el mercado busca, en lugar de lo que a mí me parecía importante.
- **Qué reforzar**: lo que más se pide y no tengo (o tengo por debajo de lo que piden), con cuántos avisos me "destrabaría" aprenderlo. Esto cambió la pregunta de "¿qué curso hago?" a "¿qué conocimiento abre más puertas en los puestos a los que apunto?".

No publico mis niveles: el punto es el método. Cruzar lo que piden los avisos (sobre todo lo excluyente) con lo que uno sabe convierte la búsqueda en un plan de estudio concreto.

## 5. Inglés, modalidad y geografía

Busco desde Mar del Plata, así que dos preguntas me importan: **cuánto hay en mi ciudad** y **qué pesa en el trabajo remoto**.

In [9]:
abiertos["pide_ingles"] = abiertos["texto_n"].str.contains(r"\b(?:ingles|english)\b")
abiertos["en_ingles"] = abiertos["texto"].map(lambda s: len(re.findall(r"\b(?:the|and|with|you|our|will)\b", s.lower())) > 15)
abiertos["ingles"] = abiertos["pide_ingles"] | abiertos["en_ingles"]

mod = (abiertos.groupby("modalidad")
       .agg(avisos=("id", "size"), con_ingles=("ingles", "sum"))
       .reindex(["Remoto", "Híbrido", "Presencial", "Sin dato"]).reset_index())
mod["% con inglés"] = (100 * mod["con_ingles"] / mod["avisos"]).round(0)

fig = px.bar(mod, x="modalidad", y="avisos", text="avisos",
             title="Avisos abiertos por modalidad (y cuántos piden inglés o están escritos en inglés)",
             color_discrete_sequence=["#9aa5b1"])
fig.add_bar(x=mod["modalidad"], y=mod["con_ingles"], name="con inglés", marker_color="#3b4a5a",
            text=mod["% con inglés"].astype(int).astype(str) + "%")
fig.update_layout(barmode="overlay", xaxis_title="", showlegend=False)
fig.show()

mdp = abiertos[abiertos["ubicacion"].str.contains("Mar del Plata", na=False)]
otro_pais = mdp["motivos"].map(lambda ms: any("otro pais" in norm(m) for m in ms))
mdp_real = mdp[(mdp["modalidad"] != "Remoto") & ~otro_pais]
mdp_sin_ventas_ni_senior = mdp_real[~mdp_real["motivos"].map(lambda ms: any(motivo_grupo(m) in ("Es de ventas", "Es senior o de liderazgo") for m in ms))]
display(Markdown(
    f"**{int(abiertos['ingles'].mean() * 100)}%** de los avisos abiertos piden inglés o están escritos en inglés; "
    f"entre los remotos sube a **{int(mod.loc[mod.modalidad == 'Remoto', '% con inglés'].iloc[0])}%**. "
    f"Las búsquedas filtradas por Mar del Plata devolvieron casi todo trabajo remoto (y algunos avisos de otros países "
    f"mal ubicados): solo **{len(mdp_real)}** avisos abiertos son de verdad en la ciudad, y sacando ventas y puestos senior "
    f"queda{'n' if len(mdp_sin_ventas_ni_senior) != 1 else ''} **{len(mdp_sin_ventas_ni_senior)}**."
))

**77%** de los avisos abiertos piden inglés o están escritos en inglés; entre los remotos sube a **94%**. Las búsquedas filtradas por Mar del Plata devolvieron casi todo trabajo remoto (y algunos avisos de otros países mal ubicados): solo **6** avisos abiertos son de verdad en la ciudad, y sacando ventas y puestos senior queda **1**.

## 6. Lo que elegí evaluar y lo que hay

Mis evaluaciones no son aleatorias: abría lo que me parecía cercano. Comparar qué familias de puesto evalué con el veredicto que les dio la extensión muestra dónde estaba poniendo la energía y dónde me iba mejor.

In [10]:
fam = (mis.groupby(["familia de puesto", "veredicto"]).size().reset_index(name="avisos"))
orden_fam = mis["familia de puesto"].value_counts().index[::-1]
fig = px.bar(fam, x="avisos", y="familia de puesto", color="veredicto", orientation="h",
             category_orders={"familia de puesto": list(orden_fam), "veredicto": ["Postulate", "Alcanzable", "Lejos"]},
             color_discrete_map=COLOR, title=f"Mis {len(mis)} evaluaciones: familia de puesto y veredicto de la extensión")
fig.update_layout(yaxis_title="", xaxis_title="avisos", legend_title="")
fig.show()

## 7. Las reglas contra mi criterio

La extensión decide con reglas: busca años, niveles de herramientas, carreras y modalidad en el texto. Para los avisos de las búsquedas comparo su veredicto con la clasificación hecha con mi criterio (sección 1).

In [11]:
cruce = pd.crosstab(tanda["nivel_calibrado"], tanda["ext_veredicto"]).reindex(
    index=["Buena chance", "Aplicable", "No"], columns=["Postulate", "Alcanzable", "Lejos"]).fillna(0).astype(int)
fig = px.imshow(cruce, text_auto=True, color_continuous_scale=["#f4f6f8", "#3b4a5a"],
                labels={"x": "veredicto de la extensión (reglas)", "y": "clasificación con mi criterio", "color": "avisos"},
                title="¿Coinciden las reglas con mi criterio?")
fig.update_layout(height=380, coloraxis_showscale=False)
fig.show()

no_pero_post = tanda[(tanda["nivel_calibrado"] == "No") & (tanda["ext_veredicto"] == "Postulate")]
razones = (no_pero_post.explode("motivos")["motivos"].map(motivo_grupo).value_counts().head(5))
display(Markdown(
    f"La extensión le dice **Postulate** a {len(no_pero_post)} avisos que yo descarto. "
    "Las razones más comunes: " + ", ".join(f"{k.lower()} ({v})" for k, v in razones.items()) + ". "
    "Son cosas que las reglas no ven o ven distinto: si el aviso sigue abierto, si es de ventas, el nivel que publicó la empresa "
    "(que no está en el texto) y, sobre todo, **qué experiencia cuenta**: la extensión suma mis años de trabajo en general, "
    "mientras que mi criterio pide experiencia en esa tarea puntual."
))

La extensión le dice **Postulate** a 167 avisos que yo descarto. Las razones más comunes: es senior o de liderazgo (64), pide más años de experiencia (45), el aviso ya cerró (40), es de ventas (35), ubicación o país (28). Son cosas que las reglas no ven o ven distinto: si el aviso sigue abierto, si es de ventas, el nivel que publicó la empresa (que no está en el texto) y, sobre todo, **qué experiencia cuenta**: la extensión suma mis años de trabajo en general, mientras que mi criterio pide experiencia en esa tarea puntual.

## 8. Lo que me llevo

Una parte de las conclusiones sale de los números de arriba, y otra de cómo cambió mi forma de buscar:

- **Buscar "nivel inicial" no alcanza.** Una parte grande de lo que se publica como inicial pide experiencia que alguien que arranca no tiene. Filtrar por texto, y no por la etiqueta, me ahorra tiempo.
- **Las credenciales pesan tanto como las tareas.** Muchos avisos de análisis listan carreras económicas o de ingeniería aunque el trabajo sea leer datos y armar reportes. Las ciencias sociales casi no aparecen. Eso no me cierra la puerta, pero me dice que tengo que mostrar qué sé hacer (proyectos, portfolio) y no solo qué estudié.
- **El inglés abre más que cualquier herramienta.** Sobre todo en el trabajo remoto, que es la mayoría de lo que veo desde Mar del Plata.
- **La extensión sirve para dos cosas además de filtrar.** Me dice qué destacar en el currículum y qué conocimientos conviene profundizar, según lo que de verdad piden los puestos a los que apunto.
- **Una herramienta ayuda, pero el criterio es mío.** Las reglas sirven para descartar rápido. Separar lo aplicable de lo que tiene chances reales necesitó que yo definiera el criterio con ejemplos marcados a mano.

### Limitaciones

- Es una muestra: mis búsquedas (análisis de datos, trabajos administrativos y recursos humanos), en LinkedIn, en unas semanas. Las búsquedas de LinkedIn interpretan las palabras clave y devuelven resultados amplios.
- La detección de años, carreras e idioma se hace con expresiones regulares sobre el texto, así que puede haber falsos positivos y negativos.
- No registré de forma consistente a qué avisos me postulé, así que todavía no puedo cerrar el embudo con respuestas. Queda para una segunda parte.
- No publico los textos de los avisos ni los datos crudos: son contenido de las empresas. Muestro solo agregados.